In [7]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import time

#project imports
sys.path.append(str(Path("../src/maxcut").resolve()))

%load_ext autoreload
%autoreload 2

import config as cfg
import utils
import plotting
plotting.set_plot_style()
print(f"Figures will be saved to: {cfg.FIGURE_DIR}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Figures will be saved to: /Users/sargun/Desktop/DAMB-DAv3c/results/figures


In [13]:
# =========================================================
# Input Paths
# =========================================================
METRICS_PATH = (
    cfg.PROCESSED_DIR / "metrics.csv"
)

PATH_BASELINE_ENRICHED = (
    cfg.PROCESSED_DIR / "baseline_enriched.csv"
)
# =========================================================
# Output Paths
# =========================================================

GRAPH_WEIGHT_CONCENTRATION_PATH = (
    cfg.PROCESSED_DIR / "graph_weight_concentration_metrics.parquet"
)


In [ ]:
metrics = pd.read_csv(METRICS_PATH)

In [5]:
def parse_weights(weight_string):
    """Convert a serialized list of edge weights into a NumPy array."""
    s = str(weight_string).strip()

    if not (s.startswith("[") and s.endswith("]")):
        raise ValueError("Unexpected weight-list format")

    weights = np.fromstring(
        s[1:-1],
        sep=",",
        dtype=np.float64
    )

    if not np.isfinite(weights).all():
        raise ValueError("Non-finite edge weights encountered")

    return weights


def coverage_count(sorted_counts, q):
    """Minimum number of weight values covering at least q of edges."""
    cumulative = np.cumsum(sorted_counts)

    target = int(np.ceil(q * cumulative[-1]))

    return int(
        np.searchsorted(cumulative, target, side="left") + 1
    )


def classify_concentration(r, n90):
    if r == 1:
        return "Uniform"
    elif n90 <= 5:
        return "Concentrated"
    elif n90 <= 50:
        return "Transitional"
    else:
        return "Heterogeneous"


def compute_weight_metrics(row):
    weights = parse_weights(row["weights"])

    # Validate number of parsed edge weights
    if len(weights) != int(row["edges"]):
        raise ValueError(
            f"{row['name']}: expected {row['edges']} weights, "
            f"found {len(weights)}"
        )

    if len(weights) == 0:
        raise ValueError(f"{row['name']}: empty edge-weight list")

    # Distinct weight values and their occurrence counts
    _, counts = np.unique(
        weights,
        return_counts=True
    )

    r = len(counts)
    m = len(weights)

    # Sort frequency counts from most to least frequent
    counts = np.sort(counts)[::-1]

    n80 = coverage_count(counts, 0.80)
    n85 = coverage_count(counts, 0.85)
    n90 = coverage_count(counts, 0.90)
    n95 = coverage_count(counts, 0.95)

    return {
        "name": row["name"],
        "nodes": row["nodes"],
        "edges": row["edges"],
        "density": row["density"],
        "limit": row["limit"],
        "has_leafs": row["has_leafs"],
        "int_only": row["int_only"],

        "r": r,
        "n80": n80,
        "n85": n85,
        "n90": n90,
        "n95": n95,

        "unique_ratio": r / m,

        # Fraction of edges whose weight appears exactly once
        "singleton_edge_fraction": (
            counts[counts == 1].sum() / m
        ),

        "concentration_group": classify_concentration(r, n90)
    }

In [9]:
CHUNK_SIZE = 5

required_columns = [
    "name",
    "nodes",
    "edges",
    "density",
    "limit",
    "has_leafs",
    "int_only",
    "unique weights",
    "weights"
]

records = []
start = time.perf_counter()

for chunk_no, chunk in enumerate(
    pd.read_csv(
        METRICS_PATH,
        usecols=required_columns,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    for _, row in chunk.iterrows():
        result = compute_weight_metrics(row)

        # Validate against the original unique-weight count
        if result["r"] != int(row["unique weights"]):
            raise ValueError(
                f"{row['name']}: distinct-weight mismatch "
                f"(computed={result['r']}, "
                f"stored={row['unique weights']})"
            )

        records.append(result)

    # Progress and checkpoint every 100 chunks
    if chunk_no % 100 == 0:
        elapsed = time.perf_counter() - start

        print(
            f"Processed {len(records)} instances | "
            f"Elapsed: {elapsed / 60:.1f} min"
        )

        pd.DataFrame(records).to_parquet(
            OUTPUT_PATH,
            index=False
        )

# Save final results
df_weight_metrics = pd.DataFrame(records)

df_weight_metrics.to_parquet(
    GRAPH_WEIGHT_CONCENTRATION_PATH,
    index=False
)

print("\nProcessing completed.")
print("Instances:", len(df_weight_metrics))
print("Shape:", df_weight_metrics.shape)
print("Saved to:", OUTPUT_PATH)

display(df_weight_metrics.head())

Processed 500 instances | Elapsed: 0.4 min
Processed 1000 instances | Elapsed: 0.7 min
Processed 1500 instances | Elapsed: 1.0 min
Processed 2000 instances | Elapsed: 1.4 min
Processed 2500 instances | Elapsed: 1.9 min
Processed 3000 instances | Elapsed: 2.3 min
Processed 3500 instances | Elapsed: 2.8 min

Processing completed.
Instances: 3506
Shape: (3506, 15)
Saved to: /Users/sargun/Desktop/DAMB-DAv3c/data/processed/graph_weight_concentration_metrics.parquet


,name,nodes,edges,density,limit,has_leafs,int_only,r,n80,n85,n90,n95,unique_ratio,singleton_edge_fraction,concentration_group
0,G55,4969,12498,0.001013,13.945,True,True,1,1,1,1,1,0.000080,0.0,Uniform
1,G56,4969,12498,0.001013,13.922,True,True,2,2,2,2,2,0.000160,0.0,Concentrated
2,G57,5000,10000,0.000800,11.299,False,True,2,2,2,2,2,0.000200,0.0,Concentrated
3,G58,5000,29570,0.002366,11.876,False,True,1,1,1,1,1,0.000034,0.0,Uniform
4,G59,5000,29570,0.002366,15.875,False,True,2,2,2,2,2,0.000068,0.0,Concentrated


In [11]:
df = df_weight_metrics.copy()

print("Total instances:", len(df))
print("Unique instance names:", df["name"].nunique())

assert df["name"].is_unique

# Coverage counts must be monotonically increasing
assert (df["n80"] <= df["n85"]).all()
assert (df["n85"] <= df["n90"]).all()
assert (df["n90"] <= df["n95"]).all()
assert (df["n95"] <= df["r"]).all()

# Every instance belongs to exactly one category
assert df["concentration_group"].notna().all()

print("All consistency checks passed.")

Total instances: 3506
Unique instance names: 3506
All consistency checks passed.


In [14]:
df_baseline = pd.read_csv(PATH_BASELINE_ENRICHED)
primary_names = set(
    df_baseline.loc[
        df_baseline["size_cat"].isin(["medium", "large"])
        & (df_baseline["limit"] > 0.25),
        "name"
    ]
)

df_int_r08 = df_weight_metrics[
    df_weight_metrics["name"].isin(primary_names)
    & df_weight_metrics["int_only"].eq(True)
].copy()

print("Integer medium + large instances:", len(df_int_r08))

category_counts = (
    df_int_r08["concentration_group"]
    .value_counts()
    .reindex(
        [
            "Uniform",
            "Concentrated",
            "Transitional",
            "Heterogeneous"
        ],
        fill_value=0
    )
)

display(category_counts)

Integer medium + large instances: 349


concentration_group
Uniform           49
Concentrated     123
Transitional      73
Heterogeneous    104
Name: count, dtype: int64

In [15]:
df_nonint = df_weight_metrics[
    df_weight_metrics["name"].isin(primary_names)
    & df_weight_metrics["int_only"].eq(False)
].copy()

summary = pd.DataFrame({
    "metric": [
        "Distinct-value ratio > 95%",
        "Singleton-edge fraction > 95%"
    ],
    "instances": [
        (df_nonint["unique_ratio"] > 0.95).sum(),
        (df_nonint["singleton_edge_fraction"] > 0.95).sum()
    ],
    "total": len(df_nonint)
})

summary["percentage"] = (
    100 * summary["instances"] / summary["total"]
)

display(summary.round(2))

display(
    df_nonint[
        ["unique_ratio", "singleton_edge_fraction"]
    ].describe()
)

,metric,instances,total,percentage
0,Distinct-value ratio > 95%,364,389,93.57
1,Singleton-edge fraction > 95%,349,389,89.72


,unique_ratio,singleton_edge_fraction
count,3.890000e+02,389.000000
mean,9.650803e-01,0.961453
std,1.665191e-01,0.167755
min,3.244343e-07,0.000000
25%,9.999892e-01,0.999978
50%,9.999979e-01,0.999996
75%,1.000000e+00,1.000000
max,1.000000e+00,1.000000
